# Legal RAG Hackathon: Main Colab

Один ноутбук для одноразового setup и повторяемого цикла: запустить нужный эксперимент, посмотреть метрики, при необходимости собрать submission.

## 1. Setup

Этот блок обычно выполняется один раз после старта или рестарта Colab.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Timur713/Legal_RAG_HSE"
REPO_DIR = Path("/content/legal-rag-hackathon")
COLAB_PATHS = "configs/paths.colab.yaml"

print(f"Repo dir: {REPO_DIR}")
print(f"Paths config: {COLAB_PATHS}")

In [ ]:
%cd /content

if not REPO_DIR.exists():
    if REPO_URL == "<PASTE_REPO_URL_HERE>":
        raise ValueError("Set REPO_URL before cloning the repository.")
    repo_name = REPO_DIR.name
    !git clone $REPO_URL $repo_name

%cd $REPO_DIR
!git pull

### 1.1 Configure Git Auth

Настраиваем `git commit` и `git push` из Colab через SSH-ключ. Все чувствительные значения вводятся интерактивно и не должны сохраняться в ноутбуке.

In [ ]:
import base64
from getpass import getpass
from urllib.parse import urlsplit

GIT_USER_NAME = getpass("Git user.name: ").strip()
GIT_USER_EMAIL = getpass("Git user.email: ").strip()
SSH_PRIVATE_KEY_B64 = getpass("Base64-encoded SSH private key for GitHub (leave empty to skip): ").strip()

if not GIT_USER_NAME or not GIT_USER_EMAIL:
    raise ValueError("Git user.name and user.email are required.")

subprocess.run(["git", "config", "user.name", GIT_USER_NAME], cwd=REPO_DIR, check=True)
subprocess.run(["git", "config", "user.email", GIT_USER_EMAIL], cwd=REPO_DIR, check=True)
print("Configured git user.name and user.email")

origin_url = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

if SSH_PRIVATE_KEY_B64:
    ssh_dir = Path.home() / ".ssh"
    ssh_dir.mkdir(parents=True, exist_ok=True)
    private_key_path = ssh_dir / "id_ed25519"
    private_key_bytes = base64.b64decode(SSH_PRIVATE_KEY_B64.encode("utf-8"))
    private_key_path.write_bytes(private_key_bytes)
    private_key_path.chmod(0o600)

    known_hosts_path = ssh_dir / "known_hosts"
    known_hosts_result = subprocess.run(
        ["ssh-keyscan", "-t", "ed25519", "github.com"],
        capture_output=True,
        text=True,
        check=True,
    )
    with known_hosts_path.open("a", encoding="utf-8") as fh:
        fh.write(known_hosts_result.stdout)

    parsed = urlsplit(origin_url)
    if parsed.scheme not in {"http", "https"} or parsed.hostname != "github.com":
        print(f"Origin uses unsupported URL for GitHub SSH conversion: {origin_url}")
    else:
        repo_path = parsed.path.lstrip("/")
        if not repo_path.endswith(".git"):
            repo_path = f"{repo_path}.git"
        ssh_origin_url = f"git@github.com:{repo_path}"
        subprocess.run(["git", "remote", "set-url", "origin", ssh_origin_url], cwd=REPO_DIR, check=True)
        ls_remote = subprocess.run(
            ["git", "ls-remote", "origin", "HEAD"],
            cwd=REPO_DIR,
            capture_output=True,
            text=True,
            check=False,
        )
        if ls_remote.returncode == 0:
            print(f"Configured SSH origin for {ssh_origin_url}")
        else:
            print("SSH origin was set, but authentication test failed:")
            print(ls_remote.stderr or ls_remote.stdout)
else:
    print("Skipped SSH key setup. Commit will work, but push may fail until origin is authenticated.")


In [ ]:
%cd $REPO_DIR
!pip install -r requirements.txt

In [ ]:
%cd $REPO_DIR
!python scripts/check_data.py --paths $COLAB_PATHS

### 1.2 Prepare Validation Splits

Фиксированные локальные сплиты для `strict_cv`, `strict_holdout` и вспомогательного `relaxed_cv` готовим один раз в setup.

In [ ]:
VALIDATION_SPLITS_CMD = f"python scripts/make_validation_splits.py --paths {COLAB_PATHS}"

print(VALIDATION_SPLITS_CMD)

In [ ]:
%cd $REPO_DIR
!$VALIDATION_SPLITS_CMD

## 2. Run Experiment + View Metrics

Меняйте параметры ниже для каждого нового эксперимента. Этот блок можно повторять сколько угодно раз.

### 2.1 Experiment: Baseline

Первый эксперимент: исходный baseline без локальной validation-обвязки.

In [ ]:
EXPERIMENT_NAME = "baseline"
EXPERIMENT_CMD = f"python scripts/run_baseline.py --paths {COLAB_PATHS} --experiment-name {EXPERIMENT_NAME}"

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.2 Experiment: Baseline Validation

Запускаем baseline не по всему train сразу, а с учетом локального validation-протокола: `strict_cv_mean`, `strict_cv_std` и `strict_holdout`.

In [ ]:
EXPERIMENT_NAME = "baseline_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--retriever tfidf --top-k 5 --extra-metric-k 20 --save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.3 Experiment: Chunked TF-IDF Validation

Независимый эксперимент для passage-level retrieval: режем документы на overlapping chunks и агрегируем score обратно в `doc_id`.

In [ ]:
EXPERIMENT_NAME = "chunked_tfidf_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--retriever chunked_tfidf --chunk-size 1600 --chunk-stride 800 --top-k 5 --extra-metric-k 20 --save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.4 Experiment: Chunked BM25 Validation

Еще один независимый passage-level эксперимент: те же overlapping chunks, но sparse scoring через BM25 вместо TF-IDF.

In [ ]:
EXPERIMENT_NAME = "chunked_bm25_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--retriever chunked_bm25 --chunk-size 1600 --chunk-stride 800 --top-k 5 --extra-metric-k 20 --save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.5 Experiment: BM25 + Lemma Validation

Document-level BM25 с лемматизацией через `pymorphy3`.

In [ ]:
EXPERIMENT_NAME = "bm25_lemma_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--retriever bm25 --top-k 5 --extra-metric-k 20 --use-lemmas --save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.6 Experiment: Chunked BM25 + Lemma Validation

Passage-level BM25 на overlapping chunks с лемматизацией.

In [ ]:
EXPERIMENT_NAME = "chunked_bm25_lemma_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--retriever chunked_bm25 --chunk-size 1600 --chunk-stride 800 --top-k 5 --extra-metric-k 20 --use-lemmas --save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.7 Experiment: TF-IDF + Lemma Validation

Document-level TF-IDF с лемматизацией через `pymorphy3`.

In [ ]:
EXPERIMENT_NAME = "tfidf_lemma_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--retriever tfidf --top-k 5 --extra-metric-k 20 --use-lemmas --save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.8 Experiment: Chunked TF-IDF + Lemma Validation

Passage-level TF-IDF на overlapping chunks с лемматизацией.

In [ ]:
EXPERIMENT_NAME = "chunked_tfidf_lemma_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--retriever chunked_tfidf --chunk-size 1600 --chunk-stride 800 --top-k 5 --extra-metric-k 20 --use-lemmas --save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.9 Experiment: Hybrid RRF BM25 + Chunked BM25 + Lemma

Гибрид document-level и passage-level BM25 с лемматизацией через Reciprocal Rank Fusion.

In [ ]:
EXPERIMENT_NAME = "hybrid_rrf_bm25_chunked_bm25_lemma_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--retriever hybrid_rrf --hybrid-retrievers bm25 chunked_bm25 --chunk-size 1600 --chunk-stride 800 --rrf-k 60 --top-k 5 --extra-metric-k 20 --use-lemmas --save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.10 Experiment: Cross-Encoder Rerank over Hybrid Sparse

First-stage retriever = `hybrid_rrf(bm25, chunked_bm25)` с лемматизацией, затем top-20 кандидатов реранжируются cross-encoder'ом по паре `question + best_chunk`.

In [ ]:
EXPERIMENT_NAME = "cross_encoder_rerank_hybrid_bm25_chunked_bm25_lemma_validation"
EXPERIMENT_CMD = (
    f"python scripts/run_baseline_validation.py "
    f"--paths {COLAB_PATHS} "
    f"--experiment-name {EXPERIMENT_NAME} "
    f"--retriever cross_encoder_rerank --top-k 5 --extra-metric-k 20 --retrieve-k 20 --rerank-top-k 20 --use-lemmas --chunk-size 1600 --chunk-stride 800 --first-stage-retriever hybrid_rrf --first-stage-hybrid-retrievers bm25 chunked_bm25 --first-stage-rrf-k 60 --reranker-model-name cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 --reranker-batch-size 16 --reranker-max-length 512 --save-test-predictions"
)

print(EXPERIMENT_CMD)

In [ ]:
%cd $REPO_DIR
!$EXPERIMENT_CMD

### 2.11 View Metrics

Смотрим метрики текущего эксперимента из сохраненного JSON.

In [ ]:
from pathlib import Path
import json

metrics_path = REPO_DIR / "outputs" / "metrics" / f"{EXPERIMENT_NAME}_metrics.json"
print(metrics_path)

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    print("Metrics file not found. Check experiment command or output naming.")

## 3. Make Submission If Needed

Запускайте этот блок только для тех экспериментов, для которых нужен сабмит. Обычно он использует `EXPERIMENT_NAME` из секции выше.

In [ ]:
SUBMISSION_CMD = f"python scripts/make_submission.py --paths {COLAB_PATHS} --experiment-name {EXPERIMENT_NAME}"

print(SUBMISSION_CMD)

In [ ]:
%cd $REPO_DIR
!$SUBMISSION_CMD

In [ ]:
submission_path = REPO_DIR / "outputs" / "submissions" / f"{EXPERIMENT_NAME}_submission.csv"
print(submission_path)

if submission_path.exists():
    import pandas as pd
    submission_df = pd.read_csv(submission_path)
    display(submission_df.head())
    print(f"rows={len(submission_df)} qids={submission_df['qid'].nunique()}")
else:
    print("Submission file not found.")

## Optional: Inspect or Save Outputs

Полезно после серии запусков, но не обязательно для каждого эксперимента.

In [ ]:
%cd $REPO_DIR
!find outputs -maxdepth 2 -type f | sort

In [ ]:
%cd $REPO_DIR
!git status

tracked_paths = ["outputs", "experiments"]
status = subprocess.run(
    ["git", "status", "--porcelain", "--", *tracked_paths],
    capture_output=True,
    text=True,
    check=False,
)

if status.stdout.strip():
    subprocess.run(["git", "add", *tracked_paths], check=True)
    subprocess.run(["git", "commit", "-m", f"add outputs for {EXPERIMENT_NAME}"], check=False)
    subprocess.run(["git", "push"], check=False)
else:
    print("Nothing new to commit in outputs/ or experiments/.")